# 3교시. 문서 구조 이해 및 추출 결과 정제

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/master/colab/03_document_structure.ipynb)

## 오늘 꼭 할 일

영수증 한 장을 PP-OCRv5로 직접 읽고, 흩어진 글자가 행으로 묶이는 과정을 확인합니다.

1. 제공 예제로 결과를 먼저 만듭니다.
2. 화면에서 이번 교시의 핵심 결과 한 가지를 확인합니다.
3. 시간이 남으면 다른 공개·비식별 자료로 반복하고 차이를 기록합니다.

**끝났다는 증거:** 화면의 `✅ 실습 완료`와
`course_outputs/clean_receipt.json` 파일

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

필수 실습에서는 제공 예제를 사용합니다. 다른 자료를 사용한 경우에는
화면에 표시된 파일명이 내가 선택한 파일과 같은지 먼저 확인합니다.
이 교시는 현재 이미지를 PP-OCRv5로 직접 읽어야 완료됩니다. 설치나
실행이 실패하면 오류를 보존하고 모델 준비 셀부터 다시 실행합니다.


## 이 노트북에서 내가 하는 일

- **필수 실습:** 현재 이미지에 PP-OCRv5를 실제 실행하고 낱말 좌표를 사람이 읽는 품목 행으로 묶습니다.
- **내가 바꾸는 곳:** 이미지 한 장을 고르고 행 묶기가 어려웠던 위치를 한 줄로 기록합니다.
- **인터넷 자료로 다시 실험:** 다른 공개·비식별 이미지 한 장으로 다시 실행해 행 묶기가 어디서 깨지는지 비교합니다.

먼저 제공 예제로 끝까지 실행해 `✅ 실습 완료`를 확인하세요. 그다음
[공개·비식별 실습 자료 찾기](https://github.com/leecks1119/document_ai_lecture/blob/master/docs/public_practice_sources.md)를 보고
입력 한 장만 바꾸어 다시 실행합니다. 2교시에서 만든 결과 파일은
3~7교시에 이어 쓸 수 있습니다. 매 교시 마지막의 **다른 자료 실험
기록**에서 잘된 점과 실패한 점을 남깁니다.

> `🟢 그대로 실행하는 셀`은 수정하지 않습니다. `🟠 내가 짧게 바꾸는
> 셀`만 필수이고, `🔵 원하면 바꾸는 셀`은 시간이 남을 때 합니다.
> 정답은 모두 공개되어 있으므로 정답을 먼저 복사하고 결과를 관찰해도 됩니다.

## 코드 셀을 읽는 방법

각 코드 셀의 맨 위에는 `코드 읽기` 주석이 있습니다.

1. `수정하지 않습니다`라고 적힌 셀은 설명을 읽고 그대로 실행합니다.
2. 주황색 필수 `TODO`만 채웁니다. 파란색 선택 `TODO`는 건너뛰어도 됩니다.
3. 실행 출력에서 `코드 읽는 법`과 `확인할 결과`를 다시 확인합니다.
4. `단계 실행 완료`가 나온 뒤 다음 코드 셀로 이동합니다.
5. 길고 어려운 준비 코드는 접혀 있습니다. 제목 왼쪽의 화살표를 눌러
   펼칠 수 있지만, 처음에는 펼치지 않아도 됩니다.

Python 문법 전체를 먼저 이해할 필요는 없습니다. 변수에 어떤 값이 들어가고,
실행 뒤 어떤 결과가 달라지는지를 중심으로 읽습니다.


In [ ]:
#@title 🟢 0. 실습 환경 준비 — 그대로 실행 { display-mode: "form" }
def _show_learning_message(markdown_text):
    try:
        from IPython.display import Markdown, display
        display(Markdown(markdown_text))
    except ImportError:
        print(markdown_text)


def show_lab_step(
    current,
    total,
    title,
    action,
    expected,
    code_help,
    edit_kind,
):
    cell_kind = {
        "required": "🟠 내가 짧게 바꾸는 셀",
        "optional": "🔵 원하면 바꾸는 셀",
        "none": "🟢 그대로 실행하는 셀",
    }[edit_kind]
    _show_learning_message(
        f"""---
### {cell_kind} · {current}/{total} · {title}

**지금 할 일:** {action}

**코드 읽는 법:** {code_help}

**이 단계에서 확인할 결과:** {expected}
"""
    )


def complete_lab_step(current, total, expected):
    next_action = (
        "결과를 확인한 뒤 다음 코드 셀을 실행하세요."
        if current < total
        else "마지막 실습 완료 문구와 산출물 파일을 확인하세요."
    )
    _show_learning_message(
        f"""> ✅ **{current}/{total} 단계 실행 완료**
>
> **결과 확인:** {expected}
>
> **다음 행동:** {next_action}
"""
    )

# ── 코드 읽기 ─────────────────────────────────────────────
# `OUTPUT_DIR`와 `load_course_assets()`를 준비합니다. 수정하지 않습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(1, 6, '공통 환경 준비', '결과 폴더와 자료 로더를 준비합니다.', '공통 작업 폴더가 표시되어야 합니다.', '`OUTPUT_DIR`와 `load_course_assets()`를 준비합니다. 수정하지 않습니다.', 'none')

import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
AUTOMATED_CHECK = os.getenv("COURSE_VALIDATE_EXAMPLE") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or AUTOMATED_CHECK:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 자료 선택에서 "
            "'제공 예제'를 고르거나 파일을 다시 선택하세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if AUTOMATED_CHECK:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())

COURSE_ASSET_BASE_URL = (
    "https://raw.githubusercontent.com/leecks1119/"
    "document_ai_lecture/master/"
)

def load_course_assets(*relative_paths):
    if AUTOMATED_CHECK:
        local_root = os.getenv("COURSE_LOCAL_ASSET_ROOT")
        if not local_root:
            raise RuntimeError(
                "자동 검증용 COURSE_LOCAL_ASSET_ROOT가 필요합니다."
            )
        root = Path(local_root)
        return {
            path: (root / path).read_bytes()
            for path in relative_paths
        }

    import requests

    loaded = {}
    missing = []
    for path in relative_paths:
        try:
            response = requests.get(
                COURSE_ASSET_BASE_URL + path,
                timeout=30,
            )
            response.raise_for_status()
            loaded[path] = response.content
        except requests.RequestException as exc:
            print(f"자동 다운로드 실패: {Path(path).name} · {exc}")
            missing.append(path)

    if missing:
        from google.colab import files

        expected = ", ".join(Path(path).name for path in missing)
        print("다음 파일을 저장소에서 내려받아 선택하세요:", expected)
        uploaded = files.upload()
        uploaded_by_name = {
            Path(name).name: content
            for name, content in uploaded.items()
        }
        for path in missing:
            filename = Path(path).name
            if filename not in uploaded_by_name:
                raise FileNotFoundError(
                    f"{filename}이 선택되지 않았습니다."
                )
            loaded[path] = uploaded_by_name[filename]

    return loaded

complete_lab_step(1, 6, '공통 작업 폴더가 표시되어야 합니다.')


In [ ]:
#@title 🔵 실습 자료 고르기 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# `실습_자료`에서 실제 OCR에 넣을 이미지 한 장만 고릅니다.
# ──────────────────────────────────────────────────────────
show_lab_step(2, 6, '입력 이미지 선택', '실제 OCR에 넣을 이미지 한 장을 고릅니다.', '선택한 원본 이미지와 파일명을 확인합니다.', '`실습_자료`에서 실제 OCR에 넣을 이미지 한 장만 고릅니다.', 'optional')

COURSE_PYTHON_PATHS = ['src/__init__.py', 'src/clean.py', 'src/export.py', 'src/extract.py', 'src/ocr.py', 'src/pipeline.py', 'src/sample_data.py', 'src/validate.py', 'src/vlm.py']
course_python_assets = load_course_assets(*COURSE_PYTHON_PATHS)
for relative_path, payload in course_python_assets.items():
    target = Path(relative_path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(payload)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# INPUT_FORM_CELL
import io
import requests
from PIL import Image
try:
    from IPython.display import display
except ImportError:
    display = print

SAMPLE_IMAGE_PATH = 'sample_docs/public_receipts/korea/taebaek_restaurant_2025_redacted.png'
sample_bytes = load_course_assets(SAMPLE_IMAGE_PATH)[SAMPLE_IMAGE_PATH]

# TODO(선택): 제공 예제 확인 뒤 입력 이미지만 바꾸어 다시 실행하세요.
실습_자료 = "제공 예제" #@param ["제공 예제", "내 컴퓨터에서 업로드", "인터넷 이미지 URL"]
인터넷_이미지_URL = "" #@param {type:"string"}
if AUTOMATED_CHECK:
    실습_자료 = "제공 예제"

input_bytes = sample_bytes
INPUT_FILE_NAME = Path(SAMPLE_IMAGE_PATH).name
if 실습_자료 == "내 컴퓨터에서 업로드":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("이미지 한 장만 선택하세요.")
    INPUT_FILE_NAME, input_bytes = next(iter(uploaded.items()))
elif 실습_자료 == "인터넷 이미지 URL":
    if not 인터넷_이미지_URL.strip():
        raise ValueError("인터넷_이미지_URL에 이미지 주소를 입력하세요.")
    response = requests.get(인터넷_이미지_URL.strip(), timeout=30)
    response.raise_for_status()
    input_bytes = response.content
    INPUT_FILE_NAME = "internet_document.png"

if len(input_bytes) > 5 * 1024 * 1024:
    raise ValueError("수업에서는 5MB 이하 이미지 한 장만 처리합니다.")
input_image = Image.open(io.BytesIO(input_bytes)).convert("RGB")
INPUT_PATH = OUTPUT_DIR / 'current_document.png'
input_image.save(INPUT_PATH)
preview = input_image.copy()
preview.thumbnail((650, 750))
display(preview)
print("선택한 자료:", 실습_자료)
print("실제 모델 입력:", INPUT_FILE_NAME, input_image.size)

complete_lab_step(2, 6, '선택한 원본 이미지와 파일명을 확인합니다.')


In [ ]:
#@title 🟢 PP-OCRv5 실제 실행 — 그대로 실행 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# `extract_with_paddleocr()`가 현재 이미지를 읽고 `reconstruct_spatial_lines()`가 좌표를 행으로
# 묶습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(3, 6, 'PP-OCRv5 실제 실행', '현재 이미지에서 글자와 좌표를 읽어 행으로 묶습니다.', '실제 모델 실행 여부·OCR 영역·복원 행을 확인합니다.', '`extract_with_paddleocr()`가 현재 이미지를 읽고 `reconstruct_spatial_lines()`가 좌표를 행으로 묶습니다.', 'none')

import importlib.metadata
import subprocess

if not AUTOMATED_CHECK:
    required = {
        "paddlepaddle": "3.2.1",
        "paddleocr": "3.7.0",
    }
    installed = {}
    for package in required:
        try:
            installed[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            installed[package] = None
    if installed != required:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "paddlepaddle==3.2.1",
            "paddleocr==3.7.0",
        ])

from src.ocr import extract_with_paddleocr, reconstruct_spatial_lines

if AUTOMATED_CHECK:
    fixture_path = "tests/fixtures/ppocrv5_recorded_receipt_tokens.json"
    tokens = json.loads(load_course_assets(fixture_path)[fixture_path])
    MODEL_EXECUTED = False
    RESULT_SOURCE = "저장된 실제 PP-OCRv5 기록 · 자동검사"
else:
    tokens = extract_with_paddleocr(INPUT_PATH)
    MODEL_EXECUTED = True
    RESULT_SOURCE = "PP-OCRv5가 현재 이미지를 직접 읽은 결과"

layout_lines = reconstruct_spatial_lines(tokens)
print("실제 모델 실행:", MODEL_EXECUTED)
print("OCR 영역:", len(tokens), "개")
print("\n--- 좌표로 다시 묶은 행 ---")
for number, line in enumerate(layout_lines, start=1):
    print(f"{number:02d}. {line}")

complete_lab_step(3, 6, '실제 모델 실행 여부·OCR 영역·복원 행을 확인합니다.')


In [ ]:
#@title 🟢 문서 구조 저장 — 그대로 실행 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# `groups`가 날짜·품목·합계 후보를 나누고 `clean_receipt.json`으로 저장합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(4, 6, '문서 구조 저장', '복원 행을 날짜·품목·합계 후보로 나누어 저장합니다.', '품목 후보와 `clean_receipt.json`을 확인합니다.', '`groups`가 날짜·품목·합계 후보를 나누고 `clean_receipt.json`으로 저장합니다.', 'none')

import re

groups = {"header": [], "date": [], "items": [], "total": [], "other": []}
cleaned_lines = [" ".join(line.split()) for line in layout_lines if line.strip()]
for line in cleaned_lines:
    if re.search(r"\d{4}[-./]\d{1,2}[-./]\d{1,2}", line):
        groups["date"].append(line)
    elif "합계" in line:
        groups["total"].append(line)
    elif re.search(r"[\d,]+\s+\d+\s+[\d,]+$", line):
        groups["items"].append(line)
    elif not groups["header"]:
        groups["header"].append(line)
    else:
        groups["other"].append(line)

clean_result = {
    "model_executed": MODEL_EXECUTED,
    "input_source": RESULT_SOURCE,
    "raw_text": "\n".join(layout_lines),
    "layout_lines": layout_lines,
    "cleaned_lines": cleaned_lines,
    "groups": groups,
    "rule": "원문에 없는 값은 추가하지 않음",
}
output_path = OUTPUT_DIR / "clean_receipt.json"
output_path.write_text(
    json.dumps(clean_result, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("\n품목 후보:", len(groups["items"]), "개")
for line in groups["items"]:
    print("-", line)
print("✅ 실습 완료:", output_path)
download_artifact(output_path)

complete_lab_step(4, 6, '품목 후보와 `clean_receipt.json`을 확인합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `내_관찰` 한 줄만 입력합니다. 전체 정답 예시는 바로 아래에 공개됩니다.
# ──────────────────────────────────────────────────────────
show_lab_step(5, 6, '내 관찰 기록', '행 묶기가 어려웠던 위치를 한 줄로 기록합니다.', '내 관찰과 전체 정답 예시를 확인합니다.', '`내_관찰` 한 줄만 입력합니다. 전체 정답 예시는 바로 아래에 공개됩니다.', 'required')

# TODO: 결과에서 행 묶기가 가장 어려웠던 곳을 한 줄로 적어 보세요.
내_관찰 = "" #@param {type:"string"}
print("내 관찰:", 내_관찰 or "아직 입력하지 않음")
print("전체 정답 예시: 품목명·단가·수량·금액이 같은 행으로 묶였는지 확인")

complete_lab_step(5, 6, '내 관찰과 전체 정답 예시를 확인합니다.')


## 선택 실험: 다른 자료로 한 번 더 확인하기

필수 실습을 먼저 끝낸 뒤, 인터넷에서 찾은 공개 문서나 개인정보를
가린 자료 한 장으로 같은 과정을 반복합니다. 결과가 잘 나오지 않아도
실패한 위치와 다음 질문을 남기면 실험이 완료됩니다.


In [ ]:
#@title 🔵 선택: 다른 자료 실험 기록 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# 이 셀은 선택 실험 기록지입니다. 위쪽 입력칸만 채우면 자료 출처, 잘된 점, 실패한 점, 다음 질문을 Markdown 파일로 저장합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(6, 6, '다른 자료 실험 기록', '인터넷에서 찾은 공개 자료나 비식별 자료의 결과를 네 줄로 정리합니다.', '`lesson03_research_note.md` 파일과 기록 내용이 표시되어야 합니다.', '이 셀은 선택 실험 기록지입니다. 위쪽 입력칸만 채우면 자료 출처, 잘된 점, 실패한 점, 다음 질문을 Markdown 파일로 저장합니다.', 'optional')

# RESEARCH_NOTE_CELL
# TODO(선택): 다른 자료로 다시 실험했다면 아래 입력칸만 채우세요.
자료_구분 = "제공 예제" #@param ["제공 예제", "공개 웹 자료", "비식별 개인 자료", "회사 승인 자료"]
자료_이름_또는_URL = "" #@param {type:"string"}
문서_종류 = "영수증" #@param ["영수증", "견적서", "신청서", "거래명세서", "표 캡처", "기타"]
잘된_점 = "" #@param {type:"string"}
실패한_점 = "" #@param {type:"string"}
다음_질문 = "" #@param {type:"string"}

research_focus = '행이나 표가 잘못 묶인 지점과 그 원인이 좌표·공백·양식 중 무엇인지 기록합니다.'
note = f'''# {문서_종류} 실험 기록

- 자료 구분: {자료_구분}
- 자료 이름 또는 원문 URL: {자료_이름_또는_URL or "미입력"}
- 이번 교시 관찰 질문: {research_focus}
- 잘된 점: {잘된_점 or "미입력"}
- 실패하거나 이상한 점: {실패한_점 or "미입력"}
- 다음에 바꿔 볼 한 가지: {다음_질문 or "미입력"}
'''
note_path = OUTPUT_DIR / "lesson03_research_note.md"
note_path.write_text(note + "\n", encoding="utf-8")
try:
    from IPython.display import Markdown, display
    display(Markdown(note))
except ImportError:
    print(note)
print("실험 기록 저장:", note_path)

complete_lab_step(6, 6, '`lesson03_research_note.md` 파일과 기록 내용이 표시되어야 합니다.')
